# German government yields

Estimate the 97.5th and 99th percentiles of the ten-year yield change on the next
available date, using historical thresholds and gradient boosting.

Data: `market_data.xlsx`, sheet `german_yields`. Yield levels are percentages;
changes and forecast thresholds are in basis points.

## 1. Load data

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
DATA_FILE = Path("market_data.xlsx")
data = pd.read_excel(DATA_FILE, sheet_name="german_yields", dtype=str)
data["date"] = pd.to_datetime(data["date"])
columns = ["date", "yield_2y", "yield_5y", "yield_10y", "yield_30y"]
data[columns[1:]] = data[columns[1:]].astype(float)

data = data.sort_values(by="date").reset_index(drop=True)

print(data.head())

print("\nRows and columns:", data.shape)
print("First date:", data["date"].min())
print("Last date:", data["date"].max())

        date  yield_2y  yield_5y  yield_10y  yield_30y
0 2000-08-01      5.14      5.20       5.30       5.49
1 2000-08-02      5.12      5.16       5.27       5.49
2 2000-08-03      5.15      5.20       5.32       5.55
3 2000-08-04      5.14      5.19       5.29       5.52
4 2000-08-07      5.09      5.15       5.29       5.52

Rows and columns: (6591, 5)
First date: 2000-08-01 00:00:00
Last date: 2026-07-14 00:00:00


## 2. Next-day target

The target is the ten-year yield change on the next available date. `target_date`
records when the outcome becomes known.

In [3]:
data["change_10y_bp"] = 100 * (data["yield_10y"] - data["yield_10y"].shift(1))
data["target_change_bp"] = data["change_10y_bp"].shift(-1)
data["target_date"] = data["date"].shift(-1)

## 3. Yield-curve features

Two slopes and the 2/5/10-year curvature, in basis points.

In [4]:
data["slope_10y_2y_bp"] = 100 * (data["yield_10y"] - data["yield_2y"])
data["slope_30y_10y_bp"] = 100 * (data["yield_30y"] - data["yield_10y"])
data["curvature_2_5_10_bp"] = 100 * (
    2 * data["yield_5y"] - data["yield_2y"] - data["yield_10y"]
)

## 4. Recent yield movements

RMS changes over the last 1, 5, 20 and 60 observations.

In [5]:
for window in [1, 5, 20, 60]:
    data[f"rv_{window}d"] = np.sqrt((data["change_10y_bp"] ** 2).rolling(window).mean())

## 5. Historical thresholds

Rolling percentiles of the last 1,000 observed yield changes.

In [6]:
data["historical_99"] = data["change_10y_bp"].rolling(1000).quantile(0.99)
data["historical_975"] = data["change_10y_bp"].rolling(1000).quantile(0.975)

## 6. Model inputs

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6591 entries, 0 to 6590
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 6591 non-null   datetime64[ns]
 1   yield_2y             6591 non-null   float64       
 2   yield_5y             6591 non-null   float64       
 3   yield_10y            6591 non-null   float64       
 4   yield_30y            6591 non-null   float64       
 5   change_10y_bp        6590 non-null   float64       
 6   target_change_bp     6590 non-null   float64       
 7   target_date          6590 non-null   datetime64[ns]
 8   slope_10y_2y_bp      6591 non-null   float64       
 9   slope_30y_10y_bp     6591 non-null   float64       
 10  curvature_2_5_10_bp  6591 non-null   float64       
 11  rv_1d                6590 non-null   float64       
 12  rv_5d                6586 non-null   float64       
 13  rv_20d               6571 non-nul

In [8]:
features = [
    "rv_1d",
    "rv_5d",
    "rv_20d",
    "rv_60d",
    "yield_2y",
    "yield_5y",
    "yield_10y",
    "yield_30y",
    "slope_10y_2y_bp",
    "slope_30y_10y_bp",
    "curvature_2_5_10_bp",
]
model_data = data.dropna().copy()
model_data.head()

,date,yield_2y,yield_5y,yield_10y,yield_30y,change_10y_bp,target_change_bp,target_date,slope_10y_2y_bp,slope_30y_10y_bp,curvature_2_5_10_bp,rv_1d,rv_5d,rv_20d,rv_60d,historical_99,historical_975
1000,2004-07-13,2.71,3.61,4.42,5.27,4.0,-6.0,2004-07-14,171.0,85.0,9.0,4.0,4.024922,5.403702,4.833908,14.0,10.0
1001,2004-07-14,2.67,3.55,4.36,5.22,-6.0,3.0,2004-07-15,169.0,86.0,7.0,6.0,4.024922,4.994997,4.895576,14.0,10.0
1002,2004-07-15,2.68,3.57,4.39,5.27,3.0,7.0,2004-07-16,171.0,88.0,7.0,3.0,4.147288,4.516636,4.909175,14.0,10.0
1003,2004-07-16,2.66,3.61,4.46,5.26,7.0,-9.0,2004-07-19,180.0,80.0,10.0,7.0,5.019960,4.224926,4.991660,14.0,10.0
1004,2004-07-19,2.62,3.53,4.37,5.21,-9.0,-2.0,2004-07-20,175.0,84.0,7.0,9.0,6.180615,4.544227,5.099020,14.0,10.0


## 7. Train, validation and test periods

- Initial training: outcomes known before 2017.
- Validation: forecast dates in 2017 to 2020.
- Test: forecast dates from 2021 onward.

Retrain at the start of each year using outcomes already known at that point.

In [9]:
cutoff_1 = pd.Timestamp("2017-01-01")
cutoff_2 = pd.Timestamp("2021-01-01")

train = model_data[model_data["target_date"] < cutoff_1].copy()
validation = model_data[
    (model_data["date"] >= cutoff_1) & (model_data["date"] < cutoff_2)
].copy()
test = model_data[model_data["date"] >= cutoff_2].copy()

## 8. Historical 99% baseline

The expected exception rate is 1%. Lower quantile loss is better.

In [10]:
boolean_v = validation["target_change_bp"] > validation["historical_99"]
boolean_t = test["target_change_bp"] > test["historical_99"]

print("Observed exceptions (validation, test):", boolean_v.sum(), boolean_t.sum())
print(
    "Expected exceptions (validation, test):", len(validation) * 0.01, len(test) * 0.01
)

Observed exceptions (validation, test): 10 20
Expected exceptions (validation, test): 10.08 14.08


In [11]:
from sklearn.metrics import mean_pinball_loss

historical_val_loss = mean_pinball_loss(
    validation["target_change_bp"], validation["historical_99"], alpha=0.99
)
historical_test_loss = mean_pinball_loss(
    test["target_change_bp"], test["historical_99"], alpha=0.99
)
print(
    "Historical quantile loss (validation, test):",
    historical_val_loss,
    historical_test_loss,
)

Historical quantile loss (validation, test): 0.1313409722222223 0.18525667613636382


## 9. Boosting: 2017 example

Fit the 99% model on the initial training data and evaluate its 2017 forecasts.

In [12]:
from sklearn.ensemble import HistGradientBoostingRegressor

boost_model = HistGradientBoostingRegressor(
    loss="quantile",
    quantile=0.99,
    learning_rate=0.04,
    max_iter=80,
    max_leaf_nodes=10,
    min_samples_leaf=100,
    l2_regularization=3.0,
    early_stopping=False,
    random_state=42,
)
boost_model.fit(train[features], train["target_change_bp"])

HistGradientBoostingRegressor(early_stopping=False, l2_regularization=3.0,
                              learning_rate=0.04, loss='quantile', max_iter=80,
                              max_leaf_nodes=10, min_samples_leaf=100,
                              quantile=0.99, random_state=42)

In [13]:
validation_2017 = validation[validation["date"] < "2018-01-01"].copy()

validation_2017["gradient_boosting_99"] = boost_model.predict(validation_2017[features])

print(validation_2017[["date", "target_change_bp", "gradient_boosting_99"]].head())

           date  target_change_bp  gradient_boosting_99
4174 2017-01-02               8.0              8.963040
4175 2017-01-03               2.0             13.631800
4176 2017-01-04               1.0              9.388679
4177 2017-01-05              -3.0              9.271462
4178 2017-01-06               5.0              9.614127


In [14]:
actual_2017 = validation_2017["target_change_bp"]
predicted_2017 = validation_2017["gradient_boosting_99"]
exceptions_2017 = actual_2017 > predicted_2017
boost_loss_2017 = mean_pinball_loss(actual_2017, predicted_2017, alpha=0.99)
print("Observed / expected exceptions:", exceptions_2017.sum(), len(actual_2017) * 0.01)
print("Quantile loss:", boost_loss_2017)

Observed / expected exceptions: 2 2.52
Quantile loss: 0.1029334961622301


## 10. Yearly 99% forecasts

Expand the training data each year, from 2017 to 2026.

In [15]:
forecasts = model_data[model_data["date"] >= cutoff_1].copy()
for year in range(2017, 2027):
    start = pd.Timestamp(year, 1, 1)
    end = pd.Timestamp(year + 1, 1, 1)
    train_year = model_data[model_data["target_date"] < start]
    rows = (forecasts["date"] >= start) & (forecasts["date"] < end)
    boost_model.fit(train_year[features], train_year["target_change_bp"])
    forecasts.loc[rows, "gradient_boosting_99"] = boost_model.predict(
        forecasts.loc[rows, features]
    )

## 11. Yearly 97.5% forecasts

Use the same settings, changing only the target percentile.

In [16]:
boost_model_975 = HistGradientBoostingRegressor(
    loss="quantile",
    quantile=0.975,
    learning_rate=0.04,
    max_iter=80,
    max_leaf_nodes=10,
    min_samples_leaf=100,
    l2_regularization=3.0,
    early_stopping=False,
    random_state=42,
)

for year in range(2017, 2027):
    start = pd.Timestamp(year, 1, 1)
    end = pd.Timestamp(year + 1, 1, 1)

    train_year = model_data[model_data["target_date"] < start]
    rows = (forecasts["date"] >= start) & (forecasts["date"] < end)

    boost_model_975.fit(train_year[features], train_year["target_change_bp"])
    forecasts.loc[rows, "gradient_boosting_975"] = boost_model_975.predict(
        forecasts.loc[rows, features]
    )

## 12. Results

Compare exception counts and quantile loss on validation and test data.

In [17]:
validation = forecasts[forecasts["date"] < cutoff_2].copy()
test = forecasts[forecasts["date"] >= cutoff_2].copy()

methods = [
    ("Historical", "historical_975", 0.975),
    ("Boosting", "gradient_boosting_975", 0.975),
    ("Historical", "historical_99", 0.99),
    ("Boosting", "gradient_boosting_99", 0.99),
]

results = []

for period, group in [("Validation", validation), ("Test", test)]:
    actual = group["target_change_bp"]

    for name, column, percentile in methods:
        predicted = group[column]
        exceptions = actual > predicted

        results.append(
            {
                "Period": period,
                "Percentile": percentile,
                "Model": name,
                "Forecasts": len(group),
                "Exceptions": exceptions.sum(),
                "Expected": len(group) * (1 - percentile),
                "Exception rate (%)": exceptions.mean() * 100,
                "Loss (bp)": mean_pinball_loss(actual, predicted, alpha=percentile),
            }
        )

comparison = pd.DataFrame(results)
print(comparison.round(6).to_string(index=False))

boost_test_loss = mean_pinball_loss(
    test["target_change_bp"], test["gradient_boosting_99"], alpha=0.99
)

improvement = 100 * (historical_test_loss - boost_test_loss) / historical_test_loss
print(f"\n99% test quantile-loss reduction: {improvement:.2f}%")

    Period  Percentile      Model  Forecasts  Exceptions  Expected  Exception rate (%)  Loss (bp)
Validation       0.975 Historical       1008          21     25.20            2.083333   0.253691
Validation       0.975   Boosting       1008          18     25.20            1.785714   0.229878
Validation       0.990 Historical       1008          10     10.08            0.992063   0.131341
Validation       0.990   Boosting       1008           5     10.08            0.496032   0.105942
      Test       0.975 Historical       1408          55     35.20            3.906250   0.365932
      Test       0.975   Boosting       1408          46     35.20            3.267045   0.325131
      Test       0.990 Historical       1408          20     14.08            1.420455   0.185257
      Test       0.990   Boosting       1408          23     14.08            1.633523   0.168009

99% test quantile-loss reduction: 9.31%
